# Modeling
## March 5, 2026

# Data Dictionary

- id - unique ID for workout
- user_id - unique ID for athlete who did the workout
- sport_type - sport for the workout (~40 unique)
- sport_type_grouped - groups workouts into main groups
- speed_mph - miles / elapsed time in hours
- distance - distance in meters
- miles - distance in miles
- kilometers - distance in kilometers
- moving_time - seconds of active moving time (pauses for red light, water break, etc)
- elapsed_time - total seconds for entire workout
- moving_minutes - minutes of active moving time
- elapsed_minutes - minutes for entire workout
- moving_time_per - moving_minutes / elapsed_minutes
- total_elevation_gain - meters of climbing
- meters_per_km - avg meters of climbing per kilometer
- feet_per_mile - avg feet of climbing per mile (for the Americans lol)
- commute - boolean flag is user marked the activity as a commute (like when Oliver bikes to class)
- manual - flag for if the workout was generated by a tracking device or if user manually entered the details
- has_gear - boolean for if user indicated which shoes/bike they used for the workout
- suffer_score - Strava metric used to describe how tough the workout is; function of heart rate and total time
- kudos_count - how many "likes" the workout received on Strava
- device_name - name used to record the workout
- start_date - date of workout
- hour - hour of workout (start)
- day_part - morning vs afternoon vs evening vs night (start)
- month - month of workout
- dayofweek - day of week of workout
- is_weekend - boolean for if dayofweek == Saturday or Sunday
- is_northern_hemisphere - start_lat > 0
- num_turns - number of turns in the GPS trace
- turns_per_mile - num_turns / miles
- wobble - how wiggly vs straight the trace is (ignoring turns)
- sprawl - distance (in miles) from most northwest vs most southeast points in the trace
- is_winter - workout in Dec-Feb for northern hemisphere, or July-August for southern
- is_summer - workout in Dec-Feb for southern hemisphere, or July-August for northern

In [7]:
import pandas as pd

df = pd.read_csv("data_for_modeling.csv")

# create winter flag
df['is_winter'] = (
    ((df['is_northern_hemisphere'] == 1) & (df['month'].isin([12, 1, 2]))) |
    ((df['is_northern_hemisphere'] == 0) & (df['month'].isin([6, 7, 8])))
).astype(int)

# create summer flag
df['is_summer'] = (
    ((df['is_northern_hemisphere'] == 1) & (df['month'].isin([6, 7, 8]))) |
    ((df['is_northern_hemisphere'] == 0) & (df['month'].isin([12, 1, 2])))
).astype(int)

### Target variable

sport_type_grouped

### Features to remove

id / user_id / sport_type / start_date / device_name / suffer_score / is_northern_hemisphere / kudos_count / has_gear / commute

### Feature Selection

* miles <- distance/miles/kilometers
* moving_time <- moving_time/moving_minutes
* elapsed_time <- elapsed_time/elapsed_minutes
* feet_per_mile <- total_elevation_gain/meters_per_km

### Binary Features

manual / is_weekend / is_northern_hemisphere

### Categorical -> One-hot encoding

day_part

### GPS features -> combine into binary variable(has_gps)
⁠num_turns / turns_per_mile / wobble / sprawl



In [8]:
df['has_gps'] = df['num_turns'].notna().astype(int)

gps_cols = ['num_turns','turns_per_mile','wobble','sprawl']
df[gps_cols] = df[gps_cols].fillna(0)

df['has_gps'].value_counts()

has_gps
1    342829
0     55084
Name: count, dtype: int64

In [9]:
# Remove Anomalies
df = df[df['is_anomaly']==0].copy()

In [10]:
# --- Remove leakage-prone features and save updated dataset ---
leak_cols = ["kudos_count", "has_gear", "commute"]

df_no_leak = df.drop(columns=existing).copy()

print("Dropped. New shape:", df_no_leak.shape)
print("Any leakage cols left in df_no_leak?", set(leak_cols).intersection(df_no_leak.columns))

Dropped. New shape: (397913, 34)
Any leakage cols left in df_no_leak? set()


In [11]:
out_path = "data_for_modeling_v2.csv"
df_no_leak.to_csv(out_path, index=False)
print(f"Saved cleaned dataset to: {out_path}")

Saved cleaned dataset to: data_for_modeling_v2.csv


In [13]:
df = pd.read_csv("data_for_modeling_v2.csv")

In [14]:
# Define Target and Features
y = df['sport_type_grouped']

final_feature_list = [
    # Core workout intensity / size
    "speed_mph",
    "miles",
    "moving_time",
    "elapsed_time",
    "moving_time_per",
    "feet_per_mile",

    # Route / GPS-shape features (+ indicator)
    "has_gps",
    "num_turns",
    "turns_per_mile",
    "wobble",
    "sprawl",

    # Time patterns
    "hour",
    "month",
    "dayofweek",
    "is_weekend",

    # Context flags
    "manual",
    "is_winter",
    "is_summer",


    # Categorical (we will one-hot encode later)
    "day_part",
]

X = df[final_feature_list].copy()


In [15]:
# Leakage Sanity Check
leak_cols = {"id", "user_id", "sport_type", "sport_type_grouped", "start_date", "device_name", "suffer_score", "is_anomaly"}
print("Leakage cols in X:", leak_cols.intersection(X.columns))

# Confirm no missing values remain after GPS filling
print(X.isna().sum().sort_values(ascending=False).head(10))


Leakage cols in X: set()
speed_mph     0
sprawl        0
is_summer     0
is_winter     0
manual        0
is_weekend    0
dayofweek     0
month         0
hour          0
wobble        0
dtype: int64


In [16]:
# Encode categorical variables
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify catergorical + numeric columns
categorical_cols = ["day_part"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# Preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

In [17]:
# Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
sport_type_grouped
Ride       0.419841
Run        0.366082
Workout    0.093403
Walk       0.091773
Hike       0.028901
Name: proportion, dtype: float64

Test distribution:
sport_type_grouped
Ride       0.419838
Run        0.366083
Workout    0.093412
Walk       0.091766
Hike       0.028901
Name: proportion, dtype: float64


In [18]:
# Define Evaluation Metrics
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [19]:
# Scaling
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Full pipeline
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("scaling", StandardScaler(with_mean=False)),  # required after one-hot
    ("classifier", LogisticRegression(max_iter=1000))
])

In [20]:
# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1:  {macro_f1:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix (pretty)
cm = confusion_matrix(y_test, y_pred, labels=model.named_steps["classifier"].classes_)
cm_df = pd.DataFrame(
    cm,
    index=model.named_steps["classifier"].classes_,
    columns=model.named_steps["classifier"].classes_
)
print("\nConfusion Matrix:")
print(cm_df)

Accuracy: 0.8461
Macro F1:  0.7232

Classification Report:
              precision    recall  f1-score   support

        Hike       0.58      0.30      0.40      2300
        Ride       0.92      0.88      0.90     33412
         Run       0.84      0.91      0.87     29134
        Walk       0.69      0.76      0.73      7303
     Workout       0.75      0.69      0.72      7434

    accuracy                           0.85     79583
   macro avg       0.76      0.71      0.72     79583
weighted avg       0.84      0.85      0.84     79583


Confusion Matrix:
         Hike   Ride    Run  Walk  Workout
Hike      692     35    232  1196      145
Ride       10  29477   3189    80      656
Run        98   1664  26468   668      236
Walk      301    131    629  5568      674
Workout    84    791    910   521     5128


# Option. Hist Gradient Boosting

In [21]:
#Step A) Pipeline
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,   # stops when no improvement on internal validation
)

model_hgb = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", hgb)
])


In [22]:
# Step B) Fit + Evaluate
model_hgb.fit(X_train, y_train)

y_pred = model_hgb.predict(X_test)



In [23]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8994508877524094
Macro F1: 0.8184140878839556

Classification Report:
               precision    recall  f1-score   support

        Hike       0.64      0.56      0.60      2300
        Ride       0.95      0.92      0.94     33412
         Run       0.90      0.94      0.92     29134
        Walk       0.77      0.87      0.82      7303
     Workout       0.86      0.78      0.82      7434

    accuracy                           0.90     79583
   macro avg       0.83      0.81      0.82     79583
weighted avg       0.90      0.90      0.90     79583


Confusion Matrix:
 [[ 1293    20   105   845    37]
 [   39 30750  1873   120   630]
 [  173   927 27387   481   166]
 [  442    49   321  6367   124]
 [   76   481   687   406  5784]]


In [24]:
# HyperParameter Tuning
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
    "classifier__learning_rate": [0.03, 0.05, 0.08, 0.1],
    "classifier__max_depth": [None, 3, 5, 8],
    "classifier__max_leaf_nodes": [15, 31, 63, 127],
    "classifier__min_samples_leaf": [10, 20, 50, 100],
    "classifier__l2_regularization": [0.0, 0.1, 1.0],
}

search = RandomizedSearchCV(
    estimator=model_hgb,
    param_distributions=param_dist,
    n_iter=20,                 # keep small first
    scoring="f1_macro",
    cv=3,
    random_state=42,
    n_jobs=-1,                 # parallelize CV (works here)
    verbose=2
)

search.fit(X_train, y_train)

print("Best Hist GB params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)

best_model = search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("\nTEST Macro F1:", f1_score(y_test, y_pred_best, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, y_pred_best))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Hist GB params: {'classifier__min_samples_leaf': 10, 'classifier__max_leaf_nodes': 63, 'classifier__max_depth': 8, 'classifier__learning_rate': 0.1, 'classifier__l2_regularization': 1.0}
Best CV macro F1: 0.8319307310582466

TEST Macro F1: 0.8318158334688215

Classification Report:
               precision    recall  f1-score   support

        Hike       0.67      0.56      0.61      2300
        Ride       0.96      0.93      0.95     33412
         Run       0.91      0.95      0.93     29134
        Walk       0.79      0.88      0.83      7303
     Workout       0.87      0.81      0.84      7434

    accuracy                           0.91     79583
   macro avg       0.84      0.83      0.83     79583
weighted avg       0.91      0.91      0.91     79583



In [25]:
print("Best params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)


Best params: {'classifier__min_samples_leaf': 10, 'classifier__max_leaf_nodes': 63, 'classifier__max_depth': 8, 'classifier__learning_rate': 0.1, 'classifier__l2_regularization': 1.0}
Best CV macro F1: 0.8319307310582466


In [26]:
# Refit the best model on the full training set

best_hgb = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    learning_rate=0.1,
    max_depth=8,
    max_leaf_nodes=63,
    min_samples_leaf=10,
    l2_regularization=1.0
)

best_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", best_hgb)
])

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

In [27]:
print("Tuned Hist GB Accuracy:", accuracy_score(y_test, y_pred))
print("Tuned Hist GB Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=best_model.named_steps["classifier"].classes_,
                     columns=best_model.named_steps["classifier"].classes_)
print("\nConfusion Matrix:\n", cm_df)

Tuned Hist GB Accuracy: 0.9101692572534336
Tuned Hist GB Macro F1: 0.8318158334688215

Classification Report:
               precision    recall  f1-score   support

        Hike       0.67      0.56      0.61      2300
        Ride       0.96      0.93      0.95     33412
         Run       0.91      0.95      0.93     29134
        Walk       0.79      0.88      0.83      7303
     Workout       0.87      0.81      0.84      7434

    accuracy                           0.91     79583
   macro avg       0.84      0.83      0.83     79583
weighted avg       0.91      0.91      0.91     79583


Confusion Matrix:
          Hike   Ride    Run  Walk  Workout
Hike     1289     18    111   844       38
Ride       15  31123   1594    96      584
Run       144    858  27567   403      162
Walk      412     40    301  6428      122
Workout    54    392    610   351     6027


Hyperparameter tuning of the HistGradientBoosting model improved overall performance from a macro F1 of 0.84 to 0.86 on the test set. Notably, minority class performance (Hike) improved substantially while maintaining strong performance on dominant classes such as Ride and Run. This indicates better class balance and generalization without sacrificing overall accuracy.

# Option2. Gradient Boosting + RandomizedSearchCV

## Model Performance Comparison

| Model                         | Accuracy | Macro F1 | Minority Class (Hike F1) | Overall Assessment |
|--------------------------------|----------|----------|---------------------------|--------------------|
| Logistic Regression            | ~0.86    | ~0.74    | Low (~0.42 earlier)       | Underfits nonlinear patterns |
| HistGradientBoosting (Tuned)   | 0.926    | 0.858    | 0.66                      | Best balance & generalization |
| GradientBoosting (RandomSearch)| 0.926    | 0.858    | 0.66                      | Comparable to HistGB |

### Hist Gradient Boosting Classifier 
* Full RandomizedSearchCV tuning
* 500 estimators
* max_depth = 4
* learning_rate = 0.1
* subsample = 1.0
* max_features = log2

Rationale:

* Highest Macro F1
* Stable performance
* Computationally efficient
* Handles nonlinear interactions effectively


# Option3. XGBoost

In [28]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [29]:
from xgboost import XGBClassifier

In [30]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42
)

xgb_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", xgb)
])

In [31]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [33]:
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
print("XGB Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGB Macro F1:", f1_score(y_test, y_pred_xgb, average="macro"))
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

XGB Accuracy: 0.909616375356546
XGB Macro F1: 0.8306659736012211
              precision    recall  f1-score   support

           0       0.67      0.56      0.61      2300
           1       0.96      0.93      0.94     33412
           2       0.91      0.95      0.93     29134
           3       0.79      0.88      0.83      7303
           4       0.87      0.81      0.84      7434

    accuracy                           0.91     79583
   macro avg       0.84      0.82      0.83     79583
weighted avg       0.91      0.91      0.91     79583

[[ 1280    19   121   838    42]
 [   14 31091  1608    94   605]
 [  134   847 27589   409   155]
 [  418    38   311  6402   134]
 [   56   404   614   332  6028]]


In [34]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

param_dist = {
    "classifier__n_estimators": [300, 500, 800],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__max_depth": [3, 4, 6, 8],
    "classifier__min_child_weight": [1, 5, 10],
    "classifier__subsample": [0.6, 0.8, 1.0],
    "classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "classifier__reg_lambda": [0.5, 1.0, 2.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search_xgb.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('cat',
                                                                               OneHotEncoder(drop='first'),
                                                                               ['day_part']),
                                                                              ('num',
                                                                               'passthrough',
                                                                               ['speed_mph',
                                                                                'miles',
                                                                                'moving_time',
                                                                                'elapsed_time',
                                                                                'moving_time_per',
                                                                                'feet_per_mile',
                                                                                'has_gps',
                                                                                'num_turns',
                                                                                'turns_per_mile',
                                                                                'wo...
                   n_iter=20, n_jobs=-1,
                   param_distributions={'classifier__colsample_bytree': [0.6,
                                                                         0.8,
                                                                         1.0],
                                        'classifier__learning_rate': [0.03,
                                                                      0.05,
                                                                      0.1],
                                        'classifier__max_depth': [3, 4, 6, 8],
                                        'classifier__min_child_weight': [1, 5,
                                                                         10],
                                        'classifier__n_estimators': [300, 500,
                                                                     800],
                                        'classifier__reg_lambda': [0.5, 1.0,
                                                                   2.0],
                                        'classifier__subsample': [0.6, 0.8,
                                                                  1.0]},
                   random_state=42, scoring='f1_macro', verbose=2)

In [35]:
best_xgb = search_xgb.best_estimator_

y_pred_best = best_xgb.predict(X_test)

print("Best XGB params:", search_xgb.best_params_)
print("XGB TEST Accuracy:", accuracy_score(y_test, y_pred_best))
print("XGB TEST Macro F1:", f1_score(y_test, y_pred_best, average="macro"))
print(classification_report(y_test, y_pred_best))

Best XGB params: {'classifier__subsample': 0.8, 'classifier__reg_lambda': 2.0, 'classifier__n_estimators': 800, 'classifier__min_child_weight': 10, 'classifier__max_depth': 8, 'classifier__learning_rate': 0.03, 'classifier__colsample_bytree': 0.8}
XGB TEST Accuracy: 0.91525828380433
XGB TEST Macro F1: 0.8416479841354677
              precision    recall  f1-score   support

           0       0.70      0.59      0.64      2300
           1       0.96      0.93      0.95     33412
           2       0.92      0.95      0.93     29134
           3       0.80      0.88      0.84      7303
           4       0.88      0.82      0.85      7434

    accuracy                           0.92     79583
   macro avg       0.85      0.84      0.84     79583
weighted avg       0.92      0.92      0.91     79583



In [36]:
import joblib
import sklearn, xgboost

bundle = {
    "model": best_xgb,                  
    "label_encoder": le,                
    "feature_cols": X.columns.tolist(), 
    "target_col": "sport_type_grouped",
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__
}

joblib.dump(bundle, "best_xgb_bundle_v2.joblib")
print("✅ Saved: best_xgb_bundle_v2.joblib")

✅ Saved: best_xgb_bundle_v2.joblib


## Model Performance Comparison

| Model                         | Accuracy | Macro F1 | Minority Class (Hike F1) | Overall Assessment |
|--------------------------------|----------|----------|---------------------------|--------------------|
| Logistic Regression_v1         | ~0.86    | ~0.74    | Low (~0.42 earlier)       | Underfits nonlinear patterns |
| HistGradientBoosting_v2        | 0.910    | 0.832    | 0.61                      | Comparable to HistGB |
| XGBoosting_v2                  | 0.915    | 0.842    | 0.64                      | Best balance & generalization|
